# SDXL × DrawBench — Results

Here, we replicate the table 5 of the paper (DrawBench column) from our experiment results:
2×2 experiment design of **Standard vs Z-Sampling** × **SDXL vs DPO-SDXL**.

Also provides an interactive 2×2 image viewer for qualitative comparison.

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd

NOTEBOOK_DIR = Path(".")
RESULTS_DIR = NOTEBOOK_DIR / "results"
IMAGES_DIR = NOTEBOOK_DIR / "sdxl_drawbench_images"

## 1. Load and Combine Results

In [ ]:
def load_jsonl(path):
    with open(path) as f:
        return pd.DataFrame([json.loads(line) for line in f if line.strip()])

df_sdxl = load_jsonl(RESULTS_DIR / "sdxl_drawbench_eval_results.jsonl")
df_dpo = load_jsonl(RESULTS_DIR / "dpo_sdxl_drawbench_eval_results.jsonl")

print(f"SDXL records: {len(df_sdxl)}, DPO records: {len(df_dpo)}")
df_sdxl.head(2)

In [ ]:
# match paper format 
# HPS v2: ×100 (both)
# PickScore: SDXL is cosine sim [0,1], DPO is logit-scaled (~23). 
# Multiply SDXL ×100.
# AES: already correct scale (~5.4-6.1)
# IR: already correct scale (~0.5-1.8) — paper reports ×100 for display

df_sdxl["hps_standard"] *= 100
df_sdxl["hps_zsampling"] *= 100
df_sdxl["pick_standard"] *= 100
df_sdxl["pick_zsampling"] *= 100
df_sdxl["ir_standard"] *= 100
df_sdxl["ir_zsampling"] *= 100

df_dpo["hps_standard"] *= 100
df_dpo["hps_zsampling"] *= 100
df_dpo["ir_standard"] *= 100
df_dpo["ir_zsampling"] *= 100

print("After normalization:")
print(f"  SDXL HPS mean: {df_sdxl['hps_standard'].mean():.2f}")
print(f"  SDXL Pick mean: {df_sdxl['pick_standard'].mean():.2f}")
print(f"  DPO HPS mean: {df_dpo['hps_standard'].mean():.2f}")
print(f"  DPO Pick mean: {df_dpo['pick_standard'].mean():.2f}")

In [ ]:
# Merge on (prompt_idx, seed) to align SDXL and DPO results
merged = df_sdxl[["prompt_idx", "seed", "prompt"]].merge(
    df_dpo[["prompt_idx", "seed"]], on=["prompt_idx", "seed"], how="inner"
)

METRICS = ["hps", "aes", "pick", "ir"]
for m in METRICS:
    merged[f"{m}_sdxl_std"] = df_sdxl[f"{m}_standard"].values
    merged[f"{m}_sdxl_zs"] = df_sdxl[f"{m}_zsampling"].values
    merged[f"{m}_dpo_std"] = df_dpo[f"{m}_standard"].values
    merged[f"{m}_dpo_zs"] = df_dpo[f"{m}_zsampling"].values

print(f"Merged records: {len(merged)}")
merged.head(2)

## 2. Summary Table (Paper Table 5 — DrawBench)

In [ ]:
METRIC_LABELS = {"hps": "HPS v2↑", "aes": "AES↑", "pick": "PickScore↑", "ir": "IR↑"}
METHOD_LABELS = [
    ("Standard", "sdxl_std"),
    ("Z-Sampling (ours)", "sdxl_zs"),
    ("Diffusion-DPO", "dpo_std"),
    ("DPO + Z-Sampling (ours)", "dpo_zs"),
]

rows = []
for label, suffix in METHOD_LABELS:
    row = {"Method": label}
    for m in METRICS:
        col = f"{m}_{suffix}"
        row[METRIC_LABELS[m]] = merged[col].mean()
    rows.append(row)

df_table = pd.DataFrame(rows).set_index("Method")
df_table

## 3. Image Viewer (2×2 Grid)

Select a seed and prompt index

In [ ]:
# Build filename -> path index for all 4 image directories
def build_image_index(base_dir: Path) -> dict[str, Path]:
    index = {}
    for png in base_dir.rglob("*.png"):
        index[png.name] = png
    return index

idx_sdxl_std = build_image_index(IMAGES_DIR / "sdxl_drawbench")
idx_sdxl_zs = build_image_index(IMAGES_DIR / "sdxl_drawbench_zsampling")
idx_dpo_std = build_image_index(IMAGES_DIR / "dpo_sdxl_drawbench")
idx_dpo_zs = build_image_index(IMAGES_DIR / "dpo_sdxl_drawbench_zsampling")

In [ ]:
import matplotlib.pyplot as plt
from io import BytesIO
from PIL import Image
from IPython.display import display, Image as IPyImage

plt.ioff()  # never auto-display live figures; we render to static PNG ourselves

COMPARISON_DIR = NOTEBOOK_DIR / "comparison_figures"
COMPARISON_DIR.mkdir(exist_ok=True)

METHOD_SUFFIX_MAP = {
    "SDXL Standard":   "sdxl_std",
    "SDXL Z-Sampling": "sdxl_zs",
    "DPO Standard":    "dpo_std",
    "DPO + Z-Sampling": "dpo_zs",
}

def show_comparison(prompt_idx: int, seed: int, save: bool = False):
    """Display 2x2 grid of images for a given prompt and seed."""
    filenames = {
        "SDXL Standard":   (idx_sdxl_std, f"sdxl_prompt{prompt_idx:03d}_seed{seed}.png"),
        "SDXL Z-Sampling": (idx_sdxl_zs,  f"sdxl_zsample_prompt{prompt_idx:03d}_seed{seed}.png"),
        "DPO Standard":    (idx_dpo_std,   f"dpo_sdxl_prompt{prompt_idx:03d}_seed{seed}.png"),
        "DPO + Z-Sampling":(idx_dpo_zs,    f"dpo_sdxl_zsample_prompt{prompt_idx:03d}_seed{seed}.png"),
    }

    prompt_text = merged[merged["prompt_idx"] == prompt_idx]["prompt"].iloc[0]
    row = merged[(merged["prompt_idx"] == prompt_idx) & (merged["seed"] == seed)]

    fig, axes = plt.subplots(2, 2, figsize=(12, 13))
    for ax, (title, (index, fname)) in zip(axes.flat, filenames.items()):
        path = index.get(fname)
        if path and path.exists():
            ax.imshow(Image.open(path))
        else:
            ax.text(0.5, 0.5, "Not found", ha="center", va="center", transform=ax.transAxes)

        suffix = METHOD_SUFFIX_MAP[title]
        if len(row):
            r = row.iloc[0]
            metrics_str = (
                f"HPS {r[f'hps_{suffix}']:.2f}  "
                f"AES {r[f'aes_{suffix}']:.2f}  "
                f"Pick {r[f'pick_{suffix}']:.2f}  "
                f"IR {r[f'ir_{suffix}']:.2f}"
            )
            ax.set_title(f"{title}\n{metrics_str}", fontsize=12)
        else:
            ax.set_title(title, fontsize=15)

        ax.axis("off")

    plt.tight_layout()

    if save:
        out_path = COMPARISON_DIR / f"comparison_prompt{prompt_idx:03d}_seed{seed}.png"
        fig.savefig(out_path, dpi=150, bbox_inches="tight")
        print(f"Saved: {out_path}")
    else:
        fig.suptitle(f'Prompt {prompt_idx}: "{prompt_text}"', fontsize=12, y=1.01)

    buf = BytesIO()
    fig.savefig(buf, format="png", dpi=110, bbox_inches="tight")
    plt.close(fig)
    buf.seek(0)
    display(IPyImage(data=buf.getvalue()))

In [ ]:
import ipywidgets as widgets
from IPython.display import display

seed_dropdown = widgets.Dropdown(options=[42, 123, 2026], value=42, description="Seed:")
prompt_slider = widgets.IntSlider(min=0, max=199, step=1, value=0, description="Prompt:")
save_btn = widgets.Button(description="Save Figure", icon="download", button_style="success")

out = widgets.Output()

def _render(save=False):
    out.clear_output(wait=True)
    with out:
        show_comparison(prompt_slider.value, seed_dropdown.value, save=save)

def _on_change(change):
    if change["old"] != change["new"]:
        _render()

def _on_save(_):
    _render(save=True)

seed_dropdown.observe(_on_change, names="value")
prompt_slider.observe(_on_change, names="value")
save_btn.on_click(_on_save)

ui = widgets.HBox([seed_dropdown, prompt_slider, save_btn])
display(ui, out)
_render()

## 6. Difference Distribution

Show how Z-Sampling improves over the baseline.

In [ ]:
from scipy import stats

PAIRS = [
    ("SDXL: Z-S − Std",  "sdxl_zs", "sdxl_std", "#55a868"),
    ("DPO: Z-S − Std",   "dpo_zs",  "dpo_std",  "#8172b2"),
]

fig, axes = plt.subplots(2, 4, figsize=(16, 8), sharey="row")

for row, (pair_label, suf_a, suf_b, color) in enumerate(PAIRS):
    for col, m in enumerate(METRICS):
        ax = axes[row, col]
        delta = merged[f"{m}_{suf_a}"] - merged[f"{m}_{suf_b}"]

        ax.hist(delta, bins=40, color=color, alpha=0.6, edgecolor="white", linewidth=0.5)
        ax.axvline(0, color="black", linewidth=0.8, linestyle="--")
        ax.axvline(delta.mean(), color="red", linewidth=1.2, label=f"mean={delta.mean():.2f}")
        ax.axvline(delta.median(), color="orange", linewidth=1.2, linestyle=":", label=f"med={delta.median():.2f}")

        pct_pos = (delta > 0).mean() * 100
        t_stat, p_val = stats.ttest_rel(merged[f"{m}_{suf_a}"], merged[f"{m}_{suf_b}"])

        ax.set_title(f"{METRIC_LABELS[m]}" if row == 0 else "", fontsize=11)
        ax.set_xlabel(f"Δ ({pair_label})" if row == 1 else "")
        ax.legend(fontsize=8, loc="upper left")
        ax.text(0.97, 0.95, f"{pct_pos:.0f}% > 0\np={p_val:.1e}",
                transform=ax.transAxes, fontsize=8, va="top", ha="right",
                bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.8))

    axes[row, 0].set_ylabel(pair_label, fontsize=10)

plt.suptitle("Paired Differences: Z-Sampling − Standard (per prompt×seed)", fontsize=13)
plt.tight_layout()
plt.show()